# **Exercise 2 - Topic Modelling and Word Embeddings [SOLUTIONS]**

This sesison will cover concepts from lectures 2 and 3 on advanced probabilistic and algebraic representations. You will be working with implementations of topic models using the `gensim` library and get an introduction to implementations of static and contextualised word embeddings.

You are encouraged to go through the documentation of the gensim python library to get a better understanding of the implementations below: https://radimrehurek.com/gensim/intro.html

## **Topic Modelling: LDA, LSI and BERTopic**

In [ ]:
%%capture
!pip install datasets scikit-learn pyLDAvis BERTopic # let's get all the downloads and imports out of the way
!pip install "numpy==2.2.5"
!pip install "scipy==1.15.2"
!pip install gensim

**Important:** After you run the above cell, please restart the runtime to ensure that the numpy install works as intended. `Runtime -> Restart Session` or `Ctrl + M + .`

In [ ]:
# text processing
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('wordnet')
import datasets

# gensim
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel
from pprint import pprint

import tqdm
import numpy as np
import scipy
import pandas as pd

from sklearn.decomposition import TruncatedSVD

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


We will work with the simple English wikipedia again, as we did in the previous session. Load the dataset with the command `datasets.load_dataset("wikipedia", "20220301.simple", trust_remote_code=True)["train"]` (You can use a subset of 2k-5k articles for the exercise to reduce computation time).

The first step to every text processing pipeline is the tokenisation and preprocessing. Define a preprocessing function that takes a string of text and tokenises, lemmatises and removes stopwords and returns a list of tokens for further analyses. You can either use only `nltk` or combine it with preprocessing functions defined in the gensim library, for eg. `gensim.utils.simple_preprocess()` for this task.

In [ ]:
stop_words = stopwords.words('english')
lemmatiser = nltk.stem.WordNetLemmatizer

def preprocess(doc):
  doc = gensim.utils.simple_preprocess(doc, deacc=True)
  doc = [word for word in doc if word not in stop_words]
  doc = [lemmatiser().lemmatize(word) for word in doc]
  return doc

In [ ]:
wikipedia = datasets.load_dataset("wikipedia", "20220301.simple", trust_remote_code=True)["train"]
wikipedia = wikipedia.select(range(2000))
preprocessed_data = [preprocess(doc) for doc in wikipedia['text']]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

wikipedia.py:   0%|          | 0.00/36.7k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/134M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/205328 [00:00<?, ? examples/s]

You can now use your preprocessed text to create your topic models. We will compare the gensim implementations of the LDA and LSI models (see lecture slides for more theoretical details) by training them on the simple English wikipedia.

You can go through the documentation for the [LDAModel](https://radimrehurek.com/gensim/models/ldamodel.html) and the [LSIModel](https://radimrehurek.com/gensim/models/lsimodel.html). To get started with training your models, you will need the following variables (assuming `preprocessed_data` is a list of lists - with each containing list being the list of tokens in each document generated by your preprocess function):

In [ ]:
# Create Dictionary
id2word = corpora.Dictionary(preprocessed_data)

# Term Document Frequency
corpus = [id2word.doc2bow(text) for text in preprocessed_data] # returns a bag of words representation of the document

# View
print(corpus[:1]) # list of documents with each tuple being (word_id, word count in doc)

Gensim creates a unique id for each word in the document. The produced corpus shown above is a mapping of (word_id, word_frequency).

For example, (0, 1) above implies, word id 0 occurs once in the first document. Likewise, word id 1 occurs twice and so on.

This is used as the input by the LDA model.

If you want to see what word a given id corresponds to, pass the id as a key to the dictionary.

In [ ]:
print(id2word[0])

Or, you can see a human-readable form of the corpus:

In [ ]:
# Human readable format of corpus (term-frequency)
[[(id2word[id], freq) for id, freq in cp] for cp in corpus[:1]]

You can now use these to build the LDA model:

In [ ]:
# Build LDA model
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=20,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=50,
                                           alpha='auto',
                                           per_word_topics=True)

Apart from that, `alpha` and `eta` are hyperparameters that affect sparsity of the topics. According to the Gensim docs, both defaults to 1.0/num_topics prior.
`chunksize` is the number of documents to be used in each training chunk. `update_every` determines how often the model parameters should be updated and `passes` is the total number of training passes. You can try different settings to determine the ideal for your model. Try reducing or increasing the number of passes, how does that affect your model? Can you find the ideal number of topics?


The above LDA model is built with 20 different topics where each topic is a combination of keywords and each keyword contributes a certain weightage to the topic. You can examine the topics as follows:

In [ ]:
# Print the top 10 keywords in the topics
lda_model.show_topics(num_topics=20, num_words=10, formatted=False)

Topic Coherence is a convenient measure to judge how good a given topic model is. While not an absolute indicator of quality, topic coherence can give you a general idea of how 'coherent' your model is by assigning a score between 0 and 1. You can learn more here:

https://developer.ibm.com/tutorials/awb-lda-topic-modeling-text-analysis-python/#step-6-evaluate-models8

Gensim has an implementation of topic coherence we can use to evaluate our model:

In [ ]:
# Compute Coherence Score
coherence_model_lda = CoherenceModel(model=lda_model, texts=preprocessed_data, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nCoherence Score: ', coherence_lda)

Exclusively for LDA models (this does not work for LSI), you can visualise and interact with the topics your model creates using the pyLDAvis package’s interactive chart that is designed to work well with jupyter notebooks:

In [ ]:
# Plotting tools
import pyLDAvis
import pyLDAvis.gensim  # don't skip this
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Visualize the topics
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
vis

So how to infer pyLDAvis’s output?

Each bubble on the left-hand side plot represents a topic. The larger the bubble, the more prevalent is that topic.

A good topic model will have fairly big, non-overlapping bubbles scattered throughout the chart instead of being clustered in one quadrant.

A model with too many topics will typically have many overlaps, small sized bubbles clustered in one region of the chart.

If you move the cursor over one of the bubbles, the words and bars on the right-hand side will update. These words are the salient keywords that form the selected topic.

Does it look like you have a good model based on the visualisation?

**Task:** Train an LSI model (`gensim.models.lsimodel.LsiModel()`) on the simple English wikipedia and compute the coherence score. How does it compare to the coherence score of the LDA model?

In [ ]:
lsi_model = gensim.models.lsimodel.LsiModel(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=20)

In [ ]:
lsi_model.show_topics(20, num_words=10, formatted=False)

In [ ]:
# Compute Coherence Score
coherence_model_lsi = CoherenceModel(model=lsi_model, texts=preprocessed_data, dictionary=id2word, coherence='c_v')
coherence_lsi = coherence_model_lsi.get_coherence()
print('\nCoherence Score: ', coherence_lsi)

You can also assign topics to unseen documents (a 'query' in the IR context) using your models. In the retrieval context, you can find relevant documents for a query by finding the closest topic to the query, and then ranking salient documents in that topic.

In [ ]:
query = 'computers and technology'
query_bow = id2word.doc2bow(preprocess(query)) # convert to bag of words representation
query_lda = lda_model[query_bow] # get the vector rep of the query
query_lsi = lsi_model[query_bow]

In [ ]:
if query_lda and all(isinstance(item, (list, tuple)) and len(item) >= 2 for item in query_lda):
    closest_topic_lda = max(query_lda[0], key=lambda x: x[1])[0]
    print(f"The closest topic is {closest_topic_lda}")
    lda_model.show_topic(closest_topic_lda)
else:
    # Handle the case where query_lda is empty or invalid
    print("LDA model did not find any relevant topics for the query.")
    closest_topic_lda = None # Or assign a default topic

In [ ]:
if query_lsi and all(isinstance(item, (list, tuple)) and len(item) >= 2 for item in query_lsi):
    closest_topic_lsi = max(query_lsi, key=lambda x: x[1])[0]
    print(f"The closest topic is {closest_topic_lsi}")
    lsi_model.show_topic(closest_topic_lsi)
else:
    # Handle the case where query_lda is empty or invalid
    print("LSI model did not find any relevant topics for the query.")
    closest_topic_lsi = None # Or assign a default topic

We can use the following functions to retrieve relevant wikipedia articles for a given topic (since the implementations for LDA and LSI differ, we will define separate functions for each):

In [ ]:
def get_documents_for_topic_lda(topic_id, num_docs=5):
  # Get the topic distribution for each document in the corpus
  document_topic_probs = [lda_model.get_document_topics(doc) for doc in corpus]

  # Find documents where the specified topic is most dominant
  relevant_documents_indices = []
  for i, doc_topics in enumerate(document_topic_probs):
    # Find the topic with the highest probability for this document
    dominant_topic = max(doc_topics, key=lambda item: item[1])

    # If the dominant topic matches the specified topic_id, add the document index
    if dominant_topic[0] == topic_id:
      relevant_documents_indices.append(i)

  # Extract the corresponding document texts from the wikipedia dataset
  relevant_documents = [wikipedia['url'][i] for i in relevant_documents_indices]

  return relevant_documents[:num_docs]

In [ ]:
relevant_documents = get_documents_for_topic_lda(closest_topic_lda)
for i, doc in enumerate(relevant_documents):
  print(f"Document {i + 1}: {doc}")

In [ ]:
def get_documents_for_topic_lsi(topic_id, num_docs=5):

    # Get topic weights for all documents
    doc_topic_dist = np.array([np.array(lsi_model[doc])[:,1] for doc in corpus])

    # Get document indices sorted by weight for the given topic
    sorted_doc_indices = np.argsort(doc_topic_dist[:, topic_id], axis=0)[::-1]

    # Extract document URLs for the top documents
    relevant_documents = [wikipedia['url'][i] for i in sorted_doc_indices[:num_docs]]

    return relevant_documents

In [ ]:
relevant_documents = get_documents_for_topic_lsi(closest_topic_lsi)

# Print the relevant document URLs
for i, doc_url in enumerate(relevant_documents):
    print(f"Document {i + 1}: {doc_url}")

**Task:** Compare the documents returned by the LDA and LSI model on the following queries (same as last session) for a more qualitative analysis of the two models:

1. Earth's atmosphere
2. Agricultural crops
3. Parts of the human body
4. What are the official languages of countries?
5. Best places to travel

You can also experiment with your own queries. Which model do you think is better? And why?



In [ ]:
# repeat cells 27, 28, 29, 31 and 33

### Bonus: BERTopic

BERTopic uses more advanced neural representations of transformer models and c-TF-IDF for topic modelling. A bried overview of the algorithm can be found here: https://maartengr.github.io/BERTopic/algorithm/algorithm.html. To understand in more detail, please see the introductory paper by [Grootendorst(2022)](https://arxiv.org/pdf/2203.05794).

The `BERTopic` library provides an easy way to create topic models with BERT.

**Task:** Go through the `BERTopic` quickstart documentation and try out the library. Train a BERTopic model on the simple English Wikipedia, compute the coherence score and compare it to the LDA and LSI models. Run the above queries with the BERTopic model and observe the retrieved documents. Does BERTopic do better than LDA and LSI?

In [ ]:
from bertopic import BERTopic
bert = BERTopic()

topics, probs = bert.fit_transform(wikipedia['text'])

# your code here

In [ ]:
similar_topics, similarity = bert.find_topics("parts of a human body", top_n=5)

In [ ]:
similar_topics

## **Introduction to word embeddings**

You must have noticed that in the three topic models we worked with, we used different types of word representations. The LSI model uses word embeddings obtained by applying SVD dimensionality reduction on a term-document frequency matrix. BERTopic, on the other hand, uses neural word embeddings learnt by a transformer model.

Let's understand this a bit further.
We will start with a sparse matrix, which we will perform some kind of dimensionality reduction on to create dense representatins. We will build a cooccurance matrix encompassing how often individual words co-occur with all other words in the vocabulary. This should be a symmetric matrix with a dimensionality of $|V| \times |V|$, where $|V|$ is the vocabulary size.

In [ ]:
vocabulary = {}
data, row, col = [], [], []

window_size = 1 #the context window to calculate co-occurance - this can be adjusted

for tokens in tqdm.tqdm(preprocessed_data, desc="Generating vocabulary and computing coo matrix from documents."):
    for pos, token in enumerate(tokens):
        i = vocabulary.setdefault(token, len(vocabulary))
        start = max(0, pos-window_size)
        end = min(len(tokens), pos+window_size+1)
        for pos2 in range(start, end):
            if pos2 == pos:
                continue
            j = vocabulary.setdefault(tokens[pos2], len(vocabulary))
            data.append(1.)
            row.append(i)
            col.append(j)

cooccurrence_matrix = scipy.sparse.coo_matrix((data, (row, col)))
idx_to_word = {idx: word for word, idx in vocabulary.items()}

In [ ]:
# let's examine the matrix:

df = pd.DataFrame(cooccurrence_matrix.toarray(), index=vocabulary.keys(), columns=vocabulary.keys())
df

In [ ]:
# we will now perform SVD - dimensionality reduction

SVD = TruncatedSVD(n_components=100, n_iter=10, random_state=42) # you can experiment with different sizes
reduced = SVD.fit_transform(cooccurrence_matrix)

In [ ]:
# what is the shape of your matrix now?

print(reduced.shape)

You can examine the quality of your embeddings by comparing the embeddings for similar words:

In [ ]:
def most_similar(word):
  word_vector = reduced[vocabulary[word]]
  ranked_results = np.argsort(reduced @ word_vector.T)[::-1][:1]

  most_similar = [idx_to_word[i] for i in ranked_results][0]

  return most_similar


In [ ]:
print(f"Most similar word to human is {most_similar('human')}")

**Task:**

1. Try this for different words. Does your implementation return relevant results? If not, can you determine why?
2. The preprocessing step can have a big impact on the values here. Try adding in stopwords, or not doing lemmatisation or lower casing - what does that do to your results here? Does it give you better embeddings?
3. Evaluate your model on the analogies dataset, which you can load via `datasets.load_dataset("tomasmcz/word2vec_analogy")`. This dataset was introduced in [one of the original `word2vec` papers](https://arxiv.org/pdf/1310.4546.pdf), highlighting the ability of word embeddings to capture the semantic orientation of the vocabulary. For example, given an analogy such as _Berlin is to Germany as Paris is to_ ____ _?_, it is possible to perform simple vector arithmetic to retrieve the answer _France_:
$$\vec{\mathrm{Germany}} - \vec{\mathrm{Berlin}} + \vec{\mathrm{Paris}} \approx \vec{\mathrm{France}}$$
Apply this formula to the dataset, keeping track of cases where your model retrieves the correct answer. What is its overall accuracy? How many OOV tokens do you observe? (Not: you can use simple dot product similarity to compare your vectors)
4. Experiment with neural network-based word embeddings such as `word2vec` or `GLoVe`. You can load these from the gensim library. Try to calculate how well these models fare on the analogies dataset. Do they perform or worse than your model?

In [ ]:
analogies = datasets.load_dataset("tomasmcz/word2vec_analogy")['train']
analogies

In [ ]:
analogies["word_a"][0], analogies["word_b"][0], analogies["word_c"][0], analogies["word_d"][0]

In [ ]:
word1 = analogies["word_a"][8756].lower()
word2 = analogies["word_b"][8756].lower()
word3 = analogies["word_c"][8756].lower()
word4 = analogies["word_d"][8756].lower()  # Correct answer
print(word1, word2, word3, word4)

embedding1 = reduced[vocabulary[word1]]
embedding2 = reduced[vocabulary[word2]]
embedding3 = reduced[vocabulary[word3]]

if all(embedding is not None for embedding in [embedding1, embedding2, embedding3]):
  predicted_embedding = embedding2 - embedding1 + embedding3
  # Calculate cosine similarity with all words in the vocabulary
  similarities = {}
  for word, index in vocabulary.items():
      embedding4 = reduced[index]
      similarity = np.dot(predicted_embedding, embedding4) / (np.linalg.norm(predicted_embedding) * np.linalg.norm(embedding4))
      similarities[word] = similarity

  # Get the word with the highest similarity
  predicted_word = max(similarities, key=similarities.get)

In [ ]:
print(predicted_word)

In [ ]:
# to load GLoVe and word2vec

import gensim.downloader as api

# Load Google's pre-trained Word2Vec and a GloVe model (300-dimensional vectors)
word2vec = api.load("word2vec-google-news-300")
glove = api.load("glove-wiki-gigaword-300")

# Get the vector for a word
word = "king"
vector = word2vec[word]

print(f"Vector for '{word}':\n", vector)
print("Vector shape:", vector.shape)

In [ ]:
# model vocab can be accessed through:
word2vec.index_to_key

In [ ]:
word1 = analogies["word_a"][8756].lower()
word2 = analogies["word_b"][8756].lower()
word3 = analogies["word_c"][8756].lower()
word4 = analogies["word_d"][8756].lower()  # Correct answer
print(word1, word2, word3, word4)

embedding1 = word2vec[word1]
embedding2 = word2vec[word2]
embedding3 = word2vec[word3]

if all(embedding is not None for embedding in [embedding1, embedding2, embedding3]):
  predicted_embedding = embedding2 - embedding1 + embedding3
  # Calculate cosine similarity with all words in the vocabulary
  similarities = {}
  for word in word2vec.index_to_key:
      embedding4 = word2vec[word]
      similarity = np.dot(predicted_embedding, embedding4) / (np.linalg.norm(predicted_embedding) * np.linalg.norm(embedding4))
      similarities[word] = similarity

  # Get the word with the highest similarity
  predicted_word = max(similarities, key=similarities.get)

In [ ]:
print(predicted_word)

In [ ]:
word1 = analogies["word_a"][8756].lower()
word2 = analogies["word_b"][8756].lower()
word3 = analogies["word_c"][8756].lower()
word4 = analogies["word_d"][8756].lower()  # Correct answer
print(word1, word2, word3, word4)

embedding1 = glove[word1]
embedding2 = glove[word2]
embedding3 = glove[word3]

if all(embedding is not None for embedding in [embedding1, embedding2, embedding3]):
  predicted_embedding = embedding2 - embedding1 + embedding3
  # Calculate cosine similarity with all words in the vocabulary
  similarities = {}
  for word in glove.index_to_key:
      embedding4 = glove[word]
      similarity = np.dot(predicted_embedding, embedding4) / (np.linalg.norm(predicted_embedding) * np.linalg.norm(embedding4))
      similarities[word] = similarity

  # Get the word with the highest similarity
  predicted_word = max(similarities, key=similarities.get)

In [ ]:
print(predicted_word)